In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
from sklearn.model_selection import train_test_split
import torch

file_path = '/content/drive/My Drive/TUGAS AKHIR SATYA/BalineseStoryDataset/charsNamedEntity/train.xlsx'
file_path_test = '/content/drive/My Drive/TUGAS AKHIR SATYA/BalineseStoryDataset/charsNamedEntity/test.xlsx'

# Load annotated dataset
df_train = pd.read_excel(file_path)
df_test = pd.read_excel(file_path_test)
# Only keep relevant columns
df_train = df_train[['StoryID', 'StoryTitle', 'SentenceID', 'Word', 'Character Named Entity Tagset']]
df_test = df_test[['StoryID', 'StoryTitle', 'SentenceID', 'Word', 'Character Named Entity Tagset']]

In [ ]:
# Gabungkan untuk dapatkan daftar label unik dari kedua file
combined_df = pd.concat([df_train, df_test], ignore_index=True)

# Buat label2id dan id2label
unique_labels = combined_df['Character Named Entity Tagset'].unique().tolist()
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {v: k for k, v in label2id.items()}

# Fungsi bantu untuk memproses dataframe jadi list of sentences & label
def prepare_data(df):
    grouped = df.groupby(['StoryID', 'SentenceID'])
    sentences = []
    labels = []
    story_ids = []

    for (story_id, _), group in grouped:
        word_list = group['Word'].tolist()
        label_list = group['Character Named Entity Tagset'].map(label2id).tolist()
        sentences.append(word_list)
        labels.append(label_list)
        story_ids.append(story_id)

    # RETURN sebagai dictionary
    return {
        'tokens': sentences,
        'ner_tags': labels,
        'story_id': story_ids
    }


In [ ]:
from datasets import Dataset, DatasetDict

# Proses train dan test
train_data = prepare_data(df_train)
test_data = prepare_data(df_test)

# Ubah ke HuggingFace Dataset
train_dataset = Dataset.from_dict(train_data)
test_dataset = Dataset.from_dict(test_data)

# Gabungkan ke DatasetDict (opsional)
dataset = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})

model_checkpoint = "bert-base-multilingual-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.72M [00:00<?, ?B/s]

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding='max_length',
        max_length=128
    )

    all_labels = []

    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # Token pertama dari suatu kata
                label_ids.append(labels[word_idx])
            else:
                # Subword dari kata yang sama → konversi B-XXX → I-XXX
                original_label = labels[word_idx]
                original_label_name = id2label[original_label]

                if original_label_name.startswith("B-"):
                    i_label_name = "I-" + original_label_name[2:]
                    i_label_id = label2id.get(i_label_name, original_label)
                    label_ids.append(i_label_id)
                else:
                    label_ids.append(original_label)

            previous_word_idx = word_idx

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

In [ ]:
# Apply preprocessing
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

# Load model
model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=len(label2id), id2label=id2label, label2id=label2id)


Map:   0%|          | 0/4407 [00:00<?, ? examples/s]

Map:   0%|          | 0/2229 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/672M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-multilingual-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=76a62c93ea266793cabdc1a91aeb25829dbdbdc1b442ef409e6c45fbcf2c97eb
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [ ]:
from seqeval.metrics import accuracy_score, f1_score, precision_score, recall_score

def compute_metrics(p):
    predictions, labels = p
    predictions = predictions.argmax(axis=-1)

    true_labels = []
    pred_labels = []

    for true, pred in zip(labels, predictions):
        true_seq = []
        pred_seq = []
        for t, p in zip(true, pred):
            if t != -100:
                true_seq.append(id2label[t])
                pred_seq.append(id2label[p])
        true_labels.append(true_seq)
        pred_labels.append(pred_seq)

    return {
        "accuracy": accuracy_score(true_labels, pred_labels),
        "precision": precision_score(true_labels, pred_labels),
        "recall": recall_score(true_labels, pred_labels),
        "f1": f1_score(true_labels, pred_labels),
    }


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none"
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


/tmp/ipython-input-3847608069.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.256006,0.920797,0.461320,0.564529,0.507732
2,0.341700,0.196996,0.940972,0.615211,0.737635,0.670884
3,0.341700,0.179151,0.943893,0.626769,0.787094,0.697842
4,0.130400,0.161073,0.951862,0.699328,0.763910,0.730194
5,0.130400,0.178417,0.950419,0.680501,0.798300,0.734708
6,0.075700,0.178994,0.951429,0.700412,0.788640,0.741912
7,0.075700,0.182514,0.954656,0.728324,0.778980,0.752801
8,0.045300,0.209523,0.953051,0.720318,0.804096,0.759905
9,0.045300,0.209056,0.955089,0.737239,0.792504,0.763873
10,0.026900,0.209786,0.954909,0.737143,0.797527,0.766147


TrainOutput(global_step=2760, training_loss=0.11425474864849146, metrics={'train_runtime': 1296.7346, 'train_samples_per_second': 33.985, 'train_steps_per_second': 2.128, 'total_flos': 2879071263843840.0, 'train_loss': 0.11425474864849146, 'epoch': 10.0})

In [ ]:
from transformers import AutoModelForTokenClassification

# Contoh: simpan model dan tokenizer (setelah training)
model.save_pretrained("saved_model/my_model")
tokenizer.save_pretrained("saved_model/my_model")


('saved_model/my_model/tokenizer_config.json',
 'saved_model/my_model/special_tokens_map.json',
 'saved_model/my_model/vocab.txt',
 'saved_model/my_model/added_tokens.json',
 'saved_model/my_model/tokenizer.json')

In [ ]:
import shutil
from google.colab import files

# Zip
shutil.make_archive("my_model", 'zip', "saved_model/my_model")

# Unduh
files.download("my_model.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from transformers import TokenClassificationPipeline
from collections import defaultdict
import torch

# Setup pipeline untuk NER
ner_pipe = TokenClassificationPipeline(
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",  # Gabungkan B- dan I- ke satu entity_group
    device=0 if torch.cuda.is_available() else -1
)

# Bersihkan token dari subword marker
def clean_word(word):
    return word.replace("##", "").strip()

# Simpan hasil karakter berdasarkan story_id
storywise_characters = defaultdict(set)

# Proses setiap kalimat dari dataset test
for story_id, tokens in zip(dataset["test"]["story_id"], dataset["test"]["tokens"]):
    sentence = " ".join(tokens)
    preds = ner_pipe(sentence)

    entity_buffer = []
    prev_entity_type = None

    for pred in preds:
        label = pred["entity_group"]  # misalnya: PNAME, OBJ, ANM
        word = clean_word(pred["word"])

        if label in ["PNAME", "ANM", "GODS", "ADJ", "OBJ"]:  # sesuaikan dengan label karakter
            if entity_buffer:
                name = tokenizer.convert_tokens_to_string(entity_buffer).strip().title()
                if name:
                    storywise_characters[story_id].add(name)
            entity_buffer = [word]
        else:
            if entity_buffer:
                name = tokenizer.convert_tokens_to_string(entity_buffer).strip().title()
                if name:
                    storywise_characters[story_id].add(name)
                entity_buffer = []

        prev_entity_type = label

    # Flush terakhir
    if entity_buffer:
        name = tokenizer.convert_tokens_to_string(entity_buffer).strip().title()
        if name:
            storywise_characters[story_id].add(name)

# ✅ Tampilkan hasil akhir
for story, chars in storywise_characters.items():
    print(f"Story ID: {story}\nCharacters: {sorted(chars)}\n")


Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Story ID: 0
Characters: ['Cening', 'Pianakne']

Story ID: 1
Characters: ['Angsa', 'Cicing', 'I', 'I Angsa', 'I Angsa Paling', 'I Cicing Gudig', 'I Kerkuak', 'Karma', 'Kerkuak', 'Uak']

Story ID: 2
Characters: ['Anake Sane Page', 'Dedari Supraba', 'Niwatakawaca', 'Sang', 'Sang Arjuna', 'Sang Hyang Indra', 'Widiadarine']

Story ID: 3
Characters: ['Betara', 'I Ngurah Rangsasa', 'I Ngurah Sawe', 'I Riki', 'Ida Anak Agung Ngurah Rangsasa', 'Ida Ngurah Rangsasa', 'Ida Sanghyang Sambu', 'Ngurah Makaklecan', 'Ngurah Siap', 'Ngurahe', 'Paduka Betara', 'Pura Rangsasa', 'Raja Brana', 'Rangsasa']

Story ID: 4
Characters: ['Jro Panunggun Kepuh Banteng', 'Kepuhe', 'Meonge', 'Pianak Ipun', 'Ulame']

Story ID: 7
Characters: ['Bapa', 'Belibis Putih', 'I Belibis Putih', 'Kedise', 'Liu', 'Narajana', 'Pandita', 'Panditane Ane', 'Rabinne', 'Ratu Pandita']

Story ID: 8
Characters: ['Anake Luh Ento', 'Bangkene Ento', 'I Belog', 'Kurenane', 'Kurenane I Belog', 'Kurenanne', 'Meme', 'Memene I Belog', 'Memenne',

In [ ]:
import pandas as pd

# Assuming this mapping exists from earlier
story_id_to_title = df_test.groupby('StoryID')['StoryTitle'].first().to_dict()

# Flatten the results into one row per character
char_data_flat = {
    "story_id": [],
    "judul": [],
    "character": []
}

for story_id, characters in storywise_characters.items():
    for character in sorted(characters):
        char_data_flat["story_id"].append(story_id)
        char_data_flat["judul"].append(story_id_to_title.get(story_id, "Unknown"))
        char_data_flat["character"].append(character)

# Convert to DataFrame
df_flat = pd.DataFrame(char_data_flat)

# Show it
df_flat


,story_id,judul,character
0,0,anak_ririh,Cening
1,0,anak_ririh,Pianakne
2,1,angsa_teken_kerkuak,Angsa
3,1,angsa_teken_kerkuak,Cicing
4,1,angsa_teken_kerkuak,I
...,...,...,...
1465,164,yuyu_misi_enjekan_kebo_ditundune,I Pekak Dukuh
1466,164,yuyu_misi_enjekan_kebo_ditundune,I Yuyu
1467,164,yuyu_misi_enjekan_kebo_ditundune,Kekua
1468,164,yuyu_misi_enjekan_kebo_ditundune,Lutung
